In [2]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


1. LOADING & PREPROCESSING

In [3]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(df.head())
print(f"Dataset loaded with {df.shape[0]} samples and {df.shape[1]-1} features.")

   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst texture  worst perimeter  worst area  \
0             

In [5]:
missing_values = df.isnull().sum().sum()
print(f"Total missing values: {missing_values}")
X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Preprocessing complete: Data split and scaled.")

Total missing values: 0
Preprocessing complete: Data split and scaled.


1.Handling Missing Values

​Step: Verified data integrity using df.isnull().sum().

​Justification: Scikit-learn algorithms cannot process NaN (null) values. Checking ensures the model won't crash during training. Since this dataset is pre-cleaned, no further action was needed.

​2. Feature Scaling (Standardization)

​Step: Applied StandardScaler to ensure all features have a mean of 0 and a standard deviation of 1.

​Justification: This dataset has features with different ranges (e.g., "Area" is large, "Smoothness" is tiny). Without scaling, distance-based models like SVM and k-NN would ignore the smaller features, leading to poor accuracy.

3.Data Splitting

​Step: Divided the data into 80% Training and 20% Testing sets.

​Justification: This allows us to train the model on one set and evaluate it on another "unseen" set. It is the only way to check for overfitting and ensure the model works on new patient data.


2. CLASSIFICATION ALGORITHM IMPLEMENTATION 

In [6]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "k-NN": KNeighborsClassifier()
}
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    print(f"{name} Accuracy: {accuracy:.4f}")

Logistic Regression Accuracy: 0.9737
Decision Tree Accuracy: 0.9474
Random Forest Accuracy: 0.9649
SVM Accuracy: 0.9825
k-NN Accuracy: 0.9474


Logistic Regression

​How: Predicts binary class probabilities using a Sigmoid function (0 to 1).

​Why: Excellent baseline for medical diagnosis; highly interpretable and efficient for linear boundaries.

​Decision Tree

​How: Splits data through a sequence of "if-else" logic gates based on feature values.

​Why: Mimics clinical decision-making; easy to visualize which cell traits trigger a diagnosis.

​Random Forest

​How: An ensemble of many trees that votes on the final outcome.

​Why: Prevents overfitting and handles the 30 features of the breast cancer dataset with high stability.

​Support Vector Machine (SVM)

​How: Finds the optimal hyperplane that maximizes the gap between two classes.

​Why: Highly effective in high-dimensional spaces; finds precise boundaries after data scaling.

k-Nearest Neighbors (k-NN)

​How: Assigns a class based on the majority label of the closest 'k' data points.

​Why: Effective when similar patient cases (cell characteristics) naturally cluster together.

3. MODEL COMPARISON
   

In [7]:
comparison_df = pd.DataFrame(list(results.items()), columns=['Algorithm', 'Accuracy'])
comparison_df = comparison_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
print("Algorithm Performance Ranking")
print(comparison_df)
best_model = comparison_df.iloc[0]['Algorithm']
worst_model = comparison_df.iloc[-1]['Algorithm']
print(f"\nBest Performing Algorithm: {best_model} ({comparison_df.iloc[0]['Accuracy']:.4f})")
print(f"Worst Performing Algorithm: {worst_model} ({comparison_df.iloc[-1]['Accuracy']:.4f})")

Algorithm Performance Ranking
             Algorithm  Accuracy
0                  SVM  0.982456
1  Logistic Regression  0.973684
2        Random Forest  0.964912
3        Decision Tree  0.947368
4                 k-NN  0.947368

Best Performing Algorithm: SVM (0.9825)
Worst Performing Algorithm: k-NN (0.9474)


Best Performer: SVM (0.9825)

The Support Vector Machine was the most accurate model. It excels at finding the maximum-margin hyperplane, which is the optimal boundary that separates malignant and benign cases with the largest possible gap. Since we have 30 different features, SVM’s ability to handle high-dimensional data makes it a robust choice for medical classification.

Worst Performers: Decision Tree & k-NN (0.9474)
These two algorithms tied for the lowest score, likely missing the same number of samples in the test set.

​Decision Tree: While easy to understand, it often suffers from overfitting, where it learns the training data too specifically and fails to generalize to new patient data.

​k-Nearest Neighbors (k-NN): As a distance-based learner, it relies on the local similarity of data points. While accurate, it lacks the global optimization strength seen in SVM or Logistic Regression.